## Exercise 1: Create the Northbank Financial Data Model

In this exercise, you design the foundational Delta tables for Northbank Financial's analytics platform. The model includes a **customer dimension** designed for SCD Type 2 history tracking, and a **transaction fact table** optimised with liquid clustering and Change Data Feed.

After completing this exercise, return to the lab setup page to explore the tables you created in **Catalog Explorer**.

### Task 1.1 — Create the catalog and schema

Create a Unity Catalog **catalog** named `banking_lab` with an appropriate comment. Inside it, create a **schema** named `silver`.

In [ ]:
%sql
CREATE CATALOG IF NOT EXISTS banking_lab
  COMMENT 'Northbank Financial lakehouse platform for customer analytics and regulatory reporting';

CREATE SCHEMA IF NOT EXISTS banking_lab.silver
  COMMENT 'Cleansed and enriched banking data — customer dimensions and transaction facts';

### Task 1.2 — Create the `dim_customer` SCD Type 2 table

Create a managed Delta table `banking_lab.silver.dim_customer`. This table tracks **customer dimension history** using the SCD Type 2 pattern.

The table must include:

| Column | Type | Notes |
|---|---|---|
| `customer_sk` | `BIGINT GENERATED ALWAYS AS IDENTITY` | Surrogate key — auto-incrementing, unique per version |
| `customer_id` | `STRING NOT NULL` | Business/natural key from source system |
| `full_name` | `STRING` | |
| `email` | `STRING` | |
| `city` | `STRING` | City of primary residence |
| `segment` | `STRING` | Customer segment: Retail, Premium, Business |
| `account_type` | `STRING` | Account type: Current, Savings, Business |
| `valid_from` | `TIMESTAMP NOT NULL` | When this version became effective |
| `valid_to` | `TIMESTAMP NOT NULL` | When this version expired (9999-12-31 for active records) |
| `is_current` | `BOOLEAN` | True for the active version |

Apply **liquid clustering** on `customer_id` and enable **Change Data Feed**.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I create a Delta table in Databricks with GENERATED ALWAYS AS IDENTITY, CLUSTER BY, and Change Data Feed enabled in TBLPROPERTIES?"*

**Hints:**
- Use `CLUSTER BY (customer_id)` to optimise lookup performance
- Add `TBLPROPERTIES (delta.enableChangeDataFeed = true)` for audit trail support
- Use `USING DELTA` (this is the recommended format for all tables in Azure Databricks)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS banking_lab.silver.dim_customer (
    customer_sk  BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id  STRING NOT NULL,
    full_name    STRING,
    email        STRING,
    city         STRING,
    segment      STRING,
    account_type STRING,
    valid_from   TIMESTAMP NOT NULL,
    valid_to     TIMESTAMP NOT NULL,
    is_current   BOOLEAN
)
USING DELTA
CLUSTER BY (customer_id)
TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

### Task 1.3 — Create the `fact_transactions` table

Create a managed Delta table `banking_lab.silver.fact_transactions` to store individual banking payments.

| Column | Type | Notes |
|---|---|---|
| `transaction_id` | `STRING NOT NULL` | Unique transaction reference |
| `account_id` | `STRING` | Bank account identifier |
| `customer_id` | `STRING` | Links to `dim_customer` |
| `transaction_type` | `STRING` | Credit, Debit, or Transfer |
| `amount` | `DECIMAL(15,2)` | Transaction amount |
| `currency` | `STRING DEFAULT 'GBP'` | Currency code |
| `transaction_date` | `DATE` | Date of the transaction |
| `description` | `STRING` | Transaction narrative |

Apply **liquid clustering** on `customer_id, transaction_date` to support the expected query patterns (filtering by customer and/or date range). Enable **Change Data Feed** for FCA compliance auditing.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I create a Delta Lake table in Databricks SQL with liquid clustering and Change Data Feed enabled?"*

**Hints:**
- Use `CLUSTER BY (customer_id, transaction_date)` — you can specify up to 4 clustering keys
- Add `TBLPROPERTIES (delta.enableChangeDataFeed = true)`
- Use `STRING DEFAULT 'GBP'` for the currency column default value

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS banking_lab.silver.fact_transactions (
    transaction_id   STRING NOT NULL,
    account_id       STRING,
    customer_id      STRING,
    transaction_type STRING,
    amount           DECIMAL(15,2),
    currency         STRING DEFAULT 'GBP',
    transaction_date DATE,
    description      STRING
)
USING DELTA
CLUSTER BY (customer_id, transaction_date)
TBLPROPERTIES (
    delta.enableChangeDataFeed = true,
    delta.feature.allowColumnDefaults = 'supported'
);

## Exercise 2: Implement SCD Type 2 for the Customer Dimension

Northbank's regulatory reporting must reflect the **customer profile at the time of each transaction**, not just the current state. In this exercise, you load initial customer records, apply realistic attribute changes, and implement the **two-step MERGE + INSERT pattern** for SCD Type 2. You then query full customer history and perform a point-in-time lookup.

### Task 2.1 — Load the initial customer records

The cell below defines the **initial dataset** for 8 Northbank customers as a PySpark DataFrame. Your task is to **write this DataFrame to `banking_lab.silver.dim_customer`** in append mode.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I write a PySpark DataFrame to an existing Delta table in Databricks using saveAsTable in append mode?"*

**Hints:**
- Use `df.write.mode("append").saveAsTable("banking_lab.silver.dim_customer")`
- The data and DataFrame setup are already provided — add the write statement after the DataFrame is built

In [ ]:
from pyspark.sql.functions import lit, to_timestamp

initial_customers = [
    ("C-1001", "Emma Hartley",      "emma.hartley@northbank.com",     "London",     "Retail",   "Savings"),
    ("C-1002", "James Weston",      "james.weston@northbank.com",     "Manchester", "Retail",   "Current"),
    ("C-1003", "Sophia Chen",       "sophia.chen@northbank.com",      "Edinburgh",  "Premium",  "Savings"),
    ("C-1004", "Oliver Banks",      "oliver.banks@northbank.com",     "Birmingham", "Retail",   "Current"),
    ("C-1005", "Aisha Patel",       "aisha.patel@northbank.com",      "London",     "Business", "Business"),
    ("C-1006", "Liam Murray",       "liam.murray@northbank.com",      "Leeds",      "Retail",   "Savings"),
    ("C-1007", "Charlotte Wright",  "charlotte.wright@northbank.com", "Bristol",    "Premium",  "Current"),
    ("C-1008", "Noah Thompson",     "noah.thompson@northbank.com",    "Glasgow",    "Retail",   "Savings"),
]

df_initial = spark.createDataFrame(
    initial_customers,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)

df_initial = (
    df_initial
    .withColumn("valid_from",  to_timestamp(lit("2020-01-01 00:00:00")))
    .withColumn("valid_to",    to_timestamp(lit("9999-12-31 00:00:00")))
    .withColumn("is_current",  lit(True))
)

df_initial.write.mode("append").saveAsTable("banking_lab.silver.dim_customer")
print(f"Loaded {df_initial.count()} initial customer records.")

In [ ]:
%sql
-- Verify: you should see 8 customer records
SELECT customer_sk, customer_id, full_name, city, segment, is_current
FROM banking_lab.silver.dim_customer
ORDER BY customer_sk;

### Task 2.2 — Load staging data with customer changes

The operations team has received the following change notifications:

| Customer | Change |
|---|---|
| **C-1001 Emma Hartley** | Moved from **London → Oxford**, upgraded to **Premium** segment |
| **C-1003 Sophia Chen** | Relocated from **Edinburgh → London** |
| **C-1006 Liam Murray** | Updated email address only (city and segment unchanged) |
| **C-1009 Priya Singh** | **New customer** joining from Cardiff, Retail segment |

Run the cell below to create a staging DataFrame and register it as a temp view. You will use it in the next task.

In [ ]:
staging_data = [
    ("C-1001", "Emma Hartley",    "emma.hartley@northbank.com",     "Oxford",   "Premium",  "Savings"),
    ("C-1003", "Sophia Chen",     "sophia.chen@northbank.com",      "London",   "Premium",  "Savings"),
    ("C-1006", "Liam Murray",     "liam.new@northbank.com",         "Leeds",    "Retail",   "Savings"),
    ("C-1009", "Priya Singh",     "priya.singh@northbank.com",      "Cardiff",  "Retail",   "Current"),
]

df_staging = spark.createDataFrame(
    staging_data,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)
df_staging.createOrReplaceTempView("staging_customers")
print("Staging data loaded. Temp view 'staging_customers' is available.")
display(df_staging)

### Task 2.3 — Apply SCD Type 2: Step 1 — Close changed records

Write a `MERGE INTO` statement that compares `staging_customers` against the current versions in `dim_customer`. For any record where `city`, `segment`, `email`, or `account_type` has changed, **close the current version** by setting:
- `valid_to = current_timestamp()`
- `is_current = false`

> 🤖 **Databricks Assistant tip:** Ask:
> *"Write a Databricks SQL MERGE statement that closes SCD Type 2 dimension records by setting is_current=false and valid_to=current_timestamp() when any tracked attribute changes."*

**Hints:**
- Match on `target.customer_id = source.customer_id AND target.is_current = true`
- The `WHEN MATCHED AND (...)` condition should check if **any** of the tracked attributes differ
- Only update `valid_to` and `is_current` — do not overwrite the historical data columns

In [ ]:
%sql
-- Step 1: Close the current version of changed customer records
MERGE INTO banking_lab.silver.dim_customer AS target
USING staging_customers AS source
ON target.customer_id = source.customer_id
   AND target.is_current = true
WHEN MATCHED AND (
    target.city         <> source.city         OR
    target.segment      <> source.segment       OR
    target.email        <> source.email         OR
    target.account_type <> source.account_type
) THEN UPDATE SET
    target.valid_to    = current_timestamp(),
    target.is_current  = false;

### Task 2.3 — Apply SCD Type 2: Step 2 — Insert new versions and new customers

Write an `INSERT INTO` statement to:
1. Add **new current rows** for customers whose old versions were just closed (C-1001, C-1003, C-1006)
2. Add the **brand-new customer** C-1009 who has no existing record

New rows should have `valid_from = current_timestamp()`, `valid_to = TIMESTAMP '9999-12-31'`, and `is_current = true`.

> 🤖 **Databricks Assistant tip:** Ask:
> *"Write a Databricks SQL INSERT statement to add new current SCD Type 2 rows from a staging table, only for customers where no is_current=true record exists."*

**Hint:** Use a `WHERE NOT EXISTS (SELECT 1 FROM banking_lab.silver.dim_customer d WHERE d.customer_id = s.customer_id AND d.is_current = true)` to identify which staging records need a new row inserted.

In [ ]:
%sql
-- Step 2: Insert new current versions for changed customers and the new customer
INSERT INTO banking_lab.silver.dim_customer
    (customer_id, full_name, email, city, segment, account_type, valid_from, valid_to, is_current)
SELECT
    s.customer_id,
    s.full_name,
    s.email,
    s.city,
    s.segment,
    s.account_type,
    current_timestamp()              AS valid_from,
    TIMESTAMP '9999-12-31 00:00:00'  AS valid_to,
    true                             AS is_current
FROM staging_customers s
WHERE NOT EXISTS (
    SELECT 1
    FROM banking_lab.silver.dim_customer d
    WHERE d.customer_id = s.customer_id
      AND d.is_current = true
);

### Task 2.4 — Query the full version history of a customer

Write a query to show **all versions** of customer `C-1001` (Emma Hartley), ordered by `valid_from`. You should see:
- Version 1: London / Retail (expired)
- Version 2: Oxford / Premium (current)

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I query all SCD Type 2 versions for a specific customer in a Databricks Delta table?"*

**Hint:** Filter on `customer_id = 'C-1001'` and `ORDER BY valid_from ASC`.

In [ ]:
%sql
-- Show all versions of customer C-1001, ordered by valid_from
SELECT customer_sk, customer_id, full_name, city, segment, valid_from, valid_to, is_current
FROM banking_lab.silver.dim_customer
WHERE customer_id = 'C-1001'
ORDER BY valid_from ASC;

### Task 2.5 — Point-in-time customer lookup

Northbank's regulatory reporting team needs to know the **customer profile as it existed on 2020-06-15** — before any of the changes above were applied.

Write a query that returns one row per customer showing their profile on that date. Use the `valid_from` and `valid_to` columns to select the version that was active on 2020-06-15.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I write a point-in-time query on an SCD Type 2 table using valid_from and valid_to date range filtering?"*

**Hint:** A version was active on date `D` when `valid_from <= D AND valid_to > D`. Expected result: all 8 original customers in their initial state (no Oxford, no London for Sophia).

In [ ]:
%sql
-- Return the customer profile as it existed on 2020-06-15
SELECT customer_id, full_name, city, segment, account_type
FROM banking_lab.silver.dim_customer
WHERE valid_from <= TIMESTAMP '2020-06-15 00:00:00'
  AND valid_to   >  TIMESTAMP '2020-06-15 00:00:00'
ORDER BY customer_id;

## Exercise 3: Change Data Feed & FCA Audit Trail

Basel III and FCA regulations require Northbank to maintain a **queryable audit trail** of all modifications to transaction data. In this exercise, you load payment transactions, simulate data corrections (a common real-world event), and then query the **Change Data Feed** to produce a full audit log.

### Task 3.1 — Load initial transactions

Run the cell below to insert 10 payment transactions into `fact_transactions`. These represent one day's batch of processed payments.

In [ ]:
%sql
INSERT INTO banking_lab.silver.fact_transactions VALUES
  ('TXN-001', 'ACC-001', 'C-1001', 'Credit',  1500.00, 'GBP', '2025-11-01', 'Salary payment'),
  ('TXN-002', 'ACC-001', 'C-1001', 'Debit',    120.50, 'GBP', '2025-11-03', 'Amazon purchase'),
  ('TXN-003', 'ACC-002', 'C-1002', 'Debit',    250.00, 'GBP', '2025-11-05', 'Grocery store'),
  ('TXN-004', 'ACC-003', 'C-1003', 'Credit',  3200.00, 'GBP', '2025-11-07', 'Salary payment'),
  ('TXN-005', 'ACC-003', 'C-1003', 'Transfer', 500.00, 'GBP', '2025-11-10', 'Rent transfer'),
  ('TXN-006', 'ACC-004', 'C-1004', 'Debit',     89.99, 'GBP', '2025-11-12', 'Utility bill'),
  ('TXN-007', 'ACC-005', 'C-1005', 'Debit',   1500.00, 'GBP', '2025-11-14', 'Business expense'),
  ('TXN-008', 'ACC-006', 'C-1006', 'Credit',  2800.00, 'GBP', '2025-11-15', 'Salary payment'),
  ('TXN-009', 'ACC-007', 'C-1007', 'Debit',    340.00, 'GBP', '2025-11-18', 'Car insurance'),
  ('TXN-010', 'ACC-008', 'C-1008', 'Transfer', 200.00, 'GBP', '2025-11-20', 'Transfer to savings');

### Task 3.2 — Simulate transaction corrections

Northbank's reconciliation team identified two amount discrepancies. Run the cell below to apply the corrections.

In [ ]:
%sql
-- Correction 1: TXN-003 amount was mis-keyed (correct: £275.50, not £250.00)
UPDATE banking_lab.silver.fact_transactions
SET amount = 275.50, description = 'Grocery store (corrected)'
WHERE transaction_id = 'TXN-003';

-- Correction 2: TXN-007 amount is under dispute
UPDATE banking_lab.silver.fact_transactions
SET amount = 1450.00, description = 'Business expense (disputed amount)'
WHERE transaction_id = 'TXN-007';

### Task 3.3 — Query the Change Data Feed to build an audit trail

Write a query using `table_changes()` to retrieve **all change events** recorded for `fact_transactions` since the beginning of its history. Include the `_change_type`, `_commit_version`, and `_commit_timestamp` metadata columns.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I query the Change Data Feed on a Delta table in Databricks using table_changes()? Show change type and commit metadata columns."*

**Hint:** Use `table_changes('banking_lab.silver.fact_transactions', 0)` — the second argument is the starting version number.

In [ ]:
%sql
SELECT
    transaction_id,
    amount,
    description,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes('banking_lab.silver.fact_transactions', 0)
ORDER BY _commit_timestamp, transaction_id;

### Task 3.4 — Isolate corrected records for FCA reporting

For FCA reporting, the compliance team needs a view showing only the **post-correction state** of modified transactions. Filter the Change Data Feed to show only `update_postimage` records.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I filter table_changes() results in Databricks to show only update_postimage records?"*

**Hint:** Each `UPDATE` operation generates two rows: `update_preimage` (before) and `update_postimage` (after the change). Filter for `_change_type = 'update_postimage'`. Expected result: TXN-003 with £275.50 and TXN-007 with £1450.00.

In [ ]:
%sql
SELECT
    transaction_id,
    amount,
    description,
    _change_type,
    _commit_timestamp
FROM table_changes('banking_lab.silver.fact_transactions', 0)
WHERE _change_type = 'update_postimage'
ORDER BY _commit_timestamp;

## Exercise 4: Delta Lake Time Travel

Delta Lake maintains a full **transaction log** of every change made to a table. This gives you the ability to query **previous versions** of your data and even **restore** a table to an earlier state — without restoring from backup. This is critical for Northbank when transaction data is accidentally modified.

### Task 4.1 — Inspect the transaction history

Use `DESCRIBE HISTORY` to view the complete operation log of `fact_transactions`. The output shows every version of the table, including the operation type, timestamp, and user.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I view the full operation history of a Delta table in Databricks, including all version numbers?"*

Note the **version numbers** in the output — you will use them in the next tasks.

In [ ]:
%sql
DESCRIBE HISTORY banking_lab.silver.fact_transactions;

### Task 4.2 — Query original transaction amounts before corrections

Using `VERSION AS OF`, retrieve `TXN-003` and `TXN-007` **as they appeared after the initial INSERT but before the corrections** were applied.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I query a previous version of a Delta table in Databricks using VERSION AS OF in a SELECT statement?"*

**Hint:** The initial INSERT corresponds to version **1** (version 0 is the table creation). Expected result: TXN-003 with £250.00 and TXN-007 with £1500.00.

In [ ]:
%sql
SELECT transaction_id, amount, description
FROM banking_lab.silver.fact_transactions VERSION AS OF 1
WHERE transaction_id IN ('TXN-003', 'TXN-007');

### Task 4.3 — Restore the table to its pre-correction state

The reconciliation team has confirmed the corrections were applied in error and must be rolled back. Use `RESTORE TABLE` to revert `fact_transactions` to **version 1** (after the initial load, before any corrections).

After restoring, verify that TXN-003 and TXN-007 show their original amounts.

> 🤖 **Databricks Assistant tip:** Ask:
> *"How do I use RESTORE TABLE in Databricks to roll back a Delta table to a specific version number?"*

**Hint:** Use `RESTORE TABLE banking_lab.silver.fact_transactions TO VERSION AS OF 1;`

In [ ]:
%sql
RESTORE TABLE banking_lab.silver.fact_transactions TO VERSION AS OF 1;

In [ ]:
%sql
-- Verify: original amounts should be restored
SELECT transaction_id, amount, description
FROM banking_lab.silver.fact_transactions
WHERE transaction_id IN ('TXN-003', 'TXN-007');
-- Expected: TXN-003 = 250.00 'Grocery store', TXN-007 = 1500.00 'Business expense'